In [1]:
!pip install opencv-python opencv-python-headless numpy scikit-learn

  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)
   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   -- ------------------------------------- 2.4/38.9 MB 15.0 MB/s eta 0:00:03
   ------ --------------------------------- 6.3/38.9 MB 18.4 MB/s eta 0:00:02
   ----------- ---------------------------- 11.3/38.9 MB 20.7 MB/s eta 0:00:02
   ---------------- ----------------------- 16.5/38.9 MB 22.1 MB/s eta 0:00:02
   ---------------------- ----------------- 21.5/38.9 MB 22.7 MB/s eta 0:00:01
   --------------------------- ------------ 26.5/38.9 MB 23.0 MB/s eta 0:00:01
   -------------------------------- ------- 31.5/38.9 MB 23.2 MB/s eta 0:00:01
   ------------------------------------- -- 36.7/38.9 MB 23.6 MB/s eta 0:00:01
   ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV


In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier  # Adicionado conforme solicitado




# # Criação e treinamento do modelo KNeighbors
# kn_clf = KNeighborsClassifier(n_neighbors=3)
# kn_clf.fit(X_train, y_train)
#
# # Relatório de classificação para o KNeighborsClassifier
# kn_y_pred = kn_clf.predict(X_test)
# print('KNeighborsClassifier report:')
# print(metrics.classification_report(y_test, kn_y_pred))

In [7]:

from sklearn.neighbors import KNeighborsClassifier


In [8]:
import cv2
import numpy as np
import os
import glob
from sklearn.metrics.pairwise import euclidean_distances

def create_feature_vector(image):
    orb = cv2.ORB_create()
    keypoints, descriptors = orb.detectAndCompute(image, None)
    if descriptors is None:
        descriptors = np.array([])
    return descriptors

def compare_features(descriptor1, descriptor2):
    if descriptor1 is None or descriptor2 is None or len(descriptor1) == 0 or len(descriptor2) == 0:
        return float('inf')
    distances = euclidean_distances(descriptor1, descriptor2)
    return np.mean(distances)

def identify_image(input_image, dataset_path):
    input_features = create_feature_vector(input_image)
    closest_match = None
    min_distance = float('inf')

    for file_path in glob.glob(os.path.join(dataset_path, "*")):
        dataset_image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
        dataset_features = create_feature_vector(dataset_image)
        distance = compare_features(input_features, dataset_features)
        if distance < min_distance:
            min_distance = distance
            closest_match = os.path.basename(file_path)

    return closest_match


In [9]:

import glob
from sklearn.metrics.pairwise import cosine_similarity
from functools import cmp_to_key


In [11]:
# Para iniciar o algoritmo SIFT
sift = cv2.SIFT_create()


In [12]:
def compute_sift_features(image_path):
    image = cv2.imread(image_path)
    keypoints, descriptors = sift.detectAndCompute(image, None)
    return keypoints, descriptors


In [ ]:
# Calcule os descritores para todas as imagens
all_descriptors = []
for image_path in all_image_paths:
    _, descriptors = compute_sift_features(image_path)
    all_descriptors.extend(descriptors)

# Utilize o algoritmo k-means para agrupar os descritores
kmeans = KMeans(n_clusters=100)
kmeans.fit(all_descriptors)

# Cada cluster representa uma "palavra visual"


In [ ]:
def compute_histogram(image_path, kmeans):
    _, descriptors = compute_sift_features(image_path)
    labels = kmeans.predict(descriptors)
    histogram, _ = np.histogram(labels, bins=range(kmeans.n_clusters + 1))
    return histogram


In [ ]:
# Gere os histogramas para todas as imagens
all_histograms = []
for image_path in all_image_paths:
    histogram = compute_histogram(image_path, kmeans)
    all_histograms.append(histogram)

# Normalizar os histogramas para que a soma de todas as caixas seja 1 (faz com que o histograma seja invariável em escala)
scaler = StandardScaler()
all_histograms = scaler.fit_transform(all_histograms)

# Atribua rótulos a suas imagens (por exemplo, o nome da pasta pode ser o rótulo se você organizou suas imagens por pasta)
all_labels = ... # Ensure this line is properly set with valid labels
# Train and compare the SVC and KNeighborsClassifier
X_train, X_test, y_train, y_test = train_test_split(all_histograms, all_labels, test_size=0.25, random_state=1)

clf = SVC()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(metrics.classification_report(y_test, y_pred))

# Train and compare the SVC and KNeighborsClassifier
svc = SVC()
svc.fit(X_train, y_train)
svc_y_pred = svc.predict(X_test)
svc_report = metrics.classification_report(y_test, svc_y_pred, output_dict=True)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
knn_y_pred = knn.predict(X_test)
knn_report = metrics.classification_report(y_test, knn_y_pred, output_dict=True)

import pandas as pd
comparison_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-Score"],
    "SVC": [
        svc_report['weighted avg']['precision'],
        svc_report['weighted avg']['recall'],
        svc_report['weighted avg']['f1-score'],
    ],
    "KNN": [
        knn_report['weighted avg']['precision'],
        knn_report['weighted avg']['recall'],
        knn_report['weighted avg']['f1-score'],
    ]
})

print(comparison_df)

comparison_df.plot(x="Metric", y=["SVC", "KNN"], kind="bar", figsize=(10, 6), title="Model Comparison: SVC vs KNN")
